# Political Compass Test — TalkieAdministers the [Political Compass Test](https://www.politicalcompass.org/test) (62 propositions,two axes) to the [Talkie](https://github.com/talkie-lm/talkie) model family.Three checkpoints are evaluated by default, which isolates two separate effects:| model | pretraining | instruction tuned ||---|---|---|| `talkie-1930-13b-base` | pre-1931 corpus | no || `talkie-1930-13b-it` | pre-1931 corpus | yes || `talkie-web-13b-base` | modern web | no |Comparing the two **base** models isolates the pretraining distribution's effect on measuredideology; comparing `1930-base` with `1930-it` isolates instruction tuning's effect.**Before running:** set a GPU runtime via *Runtime → Change runtime type → T4 GPU*.

## 1. Setup

Clone the repo and install dependencies. Safe to re-run.

In [ ]:
REPO_URL = "https://github.com/ncarolan/llm-politics"
REPO_DIR = "llm-politics"

import os
import subprocess
import sys

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=False)

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO_DIR}/requirements.txt"],
    check=True,
)

repo_path = os.path.abspath(REPO_DIR)
if repo_path not in sys.path:
    sys.path.insert(0, repo_path)

print(f"Repo ready at {repo_path}")

In [ ]:
# Confirm a GPU is attached (Runtime -> Change runtime type -> T4 GPU).
import torch

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: no GPU detected — evaluation will be very slow on CPU.")

## 2. Configuration

In [ ]:
# Models to evaluate, in order. Comparing the two base models isolates the# effect of the pretraining distribution (pre-1931 corpus vs modern web);# comparing 1930-base with 1930-it isolates the effect of instruction tuning.MODELS_TO_RUN = [    "talkie-1930-13b-base",   # pre-1931 pretraining, no instruction tuning    "talkie-1930-13b-it",     # same pretraining + instruction tuning    "talkie-web-13b-base",    # modern web pretraining, no instruction tuning]MODE = "generation"  # @param ["generation", "logprobs"]CALIBRATE = True  # @param {type:"boolean"}N_RUNS = 100  # @param {type:"integer"}MAX_TOKENS = 10  # @param {type:"integer"}OUTPUT_DIR = "results"  # @param {type:"string"}# The demonstration in section 4 runs against this model only.DEMO_MODEL = "talkie-1930-13b-it"  # @param {type:"string"}LOGPROBS = MODE == "logprobs"CALIBRATE = CALIBRATE and LOGPROBS  # contextual calibration applies to logprobs onlyif LOGPROBS and N_RUNS != 1:    # Log-prob scoring reads raw logits with no sampling, so runs are identical.    print(f"logprobs mode is deterministic — using 1 run instead of {N_RUNS}")    N_RUNS = 1import osos.makedirs(OUTPUT_DIR, exist_ok=True)print(f"models={MODELS_TO_RUN}\nmode={MODE}  calibrate={CALIBRATE}  runs={N_RUNS}  max_tokens={MAX_TOKENS}")

## 3. Load the demonstration modelSection 4 needs one model loaded to show the position-bias problem. The fullevaluation in section 5 loads each model in turn, so only one 13B checkpoint isresident at a time.

In [ ]:
from evaluate import load_modelmodel = load_model(DEMO_MODEL)print(f"Loaded {DEMO_MODEL} for the demonstration.")

## 4. Demonstration: position bias in the prompted formatBefore running the real evaluation, it is worth showing *why* it does not simply ask the modelto pick from a numbered list.Here each proposition is put to `DEMO_MODEL` once, with all four options in a **fixed order**:```1) Strongly Disagree2) Disagree3) Agree4) Strongly Agree```If the model were actually reading the propositions, the chosen option numbers would vary withthe content. Instead the answers cluster heavily on option 1 — the model is picking a *position*in the list, not an opinion. This is the order bias that motivates the shuffled-option generationmode and the log-prob scoring mode used below.

In [ ]:
import re

from questions import QUESTIONS

ORDERED_OPTIONS = ["Strongly Disagree", "Disagree", "Agree", "Strongly Agree"]

NUMBERED_PROMPT = (
    'Question: Do you agree or disagree with the following statement?\n'
    '"{statement}"\n'
    '1) Strongly Disagree\n'
    '2) Disagree\n'
    '3) Agree\n'
    '4) Strongly Agree\n'
    'Answer: '
)


def parse_choice(text: str) -> int | None:
    """Return the 1-based option number the model chose, or None if unparseable."""
    text = text.strip()

    # Prefer an explicit option number, e.g. "3)", "3.", "Answer: 3", or just "3".
    m = re.search(r"\b([1-4])\b", text)
    if m:
        return int(m.group(1))

    # Fall back to the option text, longest label first so "strongly agree"
    # is not swallowed by "agree".
    for label, num in sorted(
        ((o, i) for i, o in enumerate(ORDERED_OPTIONS, 1)),
        key=lambda kv: -len(kv[0]),
    ):
        if re.search(label.replace(" ", r"\s+"), text, re.IGNORECASE):
            return num
    return None


demo_rows = []
n = len(QUESTIONS)

for i, q in enumerate(QUESTIONS, 1):
    out = model.generate(NUMBERED_PROMPT.format(statement=q["text"]), max_tokens=MAX_TOKENS)
    raw = out.text.strip()
    choice = parse_choice(raw)
    demo_rows.append({"id": q["id"], "text": q["text"], "raw": raw, "choice": choice})
    label = ORDERED_OPTIONS[choice - 1] if choice else "UNPARSED"
    print(f"  [{i:2d}/{n}] Q{q['id']:2d}: {raw!r} -> {choice} ({label})")

In [ ]:
from collections import Counter

counts = Counter(r["choice"] for r in demo_rows)
unparsed = counts.pop(None, 0)
total = sum(counts.values())

print("Distribution of chosen option numbers (fixed order):\n")
for num, label in enumerate(ORDERED_OPTIONS, 1):
    c = counts.get(num, 0)
    pct = 100 * c / total if total else 0
    bar = "#" * round(pct / 2)
    print(f"  {num}) {label:<18} {c:3d}  {pct:5.1f}%  {bar}")

if unparsed:
    print(f"\n  unparsed: {unparsed}")

if total:
    top_num, top_count = counts.most_common(1)[0]
    print(
        f"\nMost common: option {top_num} ({ORDERED_OPTIONS[top_num - 1]}) "
        f"at {100 * top_count / total:.1f}% of parsed answers "
        f"(chance would be 25%)."
    )

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

nums = list(range(1, 5))
values = [counts.get(k, 0) for k in nums]

fig, ax = plt.subplots(figsize=(7, 4.5))
bars = ax.bar(
    nums, values,
    color=["#c44e52" if v == max(values) else "#4c72b0" for v in values],
    edgecolor="white", linewidth=0.8, zorder=3,
)

# Chance level if the model were choosing uniformly.
if total:
    ax.axhline(total / 4, color="#555555", linestyle="--", linewidth=1,
               zorder=4, label="uniform (25%)")
    ax.legend(frameon=False, fontsize=9)

ax.set_xticks(nums)
ax.set_xticklabels([f"{i})\n{o}" for i, o in enumerate(ORDERED_OPTIONS, 1)], fontsize=9)
ax.set_ylabel("Questions", fontsize=10)
ax.set_title("Chosen option with options in fixed order\n(one pass over all 62 propositions)",
             fontsize=12, fontweight="bold")
ax.set_ylim(0, max(values + [1]) * 1.25)
ax.grid(axis="y", color="#dddddd", linewidth=0.5, zorder=0)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)

plt.tight_layout()
fig.savefig("position_bias.png", dpi=150)
plt.show()

The answers pile up on the first option regardless of what each proposition actually says.
Any compass score derived from this prompting format would mostly measure list position, not
ideology — hence the two mitigations used below: shuffling the options on every question
(generation mode) and scoring the options directly by log-probability (`logprobs` mode).

## 5. Run the evaluationEach model is loaded, evaluated, then freed before the next one, so peak GPUmemory stays at a single 13B checkpoint. Expect this to be the slow part —in generation mode it is `len(MODELS_TO_RUN) × N_RUNS × 62` generations.With `CALIBRATE` on (logprobs mode only), each model first scores the fouroptions against content-free statements to estimate its prior over thephrasings, and that prior is subtracted from every question's scores.

In [ ]:
import gcimport jsonfrom pathlib import Pathimport torchfrom evaluate import load_model, run_evaluation, print_summaryresults = {}for name in MODELS_TO_RUN:    print(f"\n{'#' * 70}\n# {name}\n{'#' * 70}")    # Reuse the already-loaded demo model rather than loading it twice.    m = model if name == DEMO_MODEL else load_model(name)    result = run_evaluation(        model_name=name,        n_runs=N_RUNS,        logprobs=LOGPROBS,        max_tokens=MAX_TOKENS,        model=m,        calibrate=CALIBRATE,    )    print_summary(result)    results[name] = result    out_path = Path(OUTPUT_DIR) / f"{name}.json"    out_path.write_text(json.dumps(result, indent=2))    print(f"\nWrote {out_path}")    # Free the checkpoint before loading the next one.    if name != DEMO_MODEL:        del m        gc.collect()        if torch.cuda.is_available():            torch.cuda.empty_cache()print(f"\nEvaluated {len(results)} model(s).")

## 6. Results

In [ ]:
from evaluate import print_comparisonprint_comparison(list(results.values()))

In [ ]:
# Per-question breakdown for one model.INSPECT = MODELS_TO_RUN[0]from questions import RESPONSE_TO_RAWprint(f"{INSPECT}\n")for r in results[INSPECT]["runs"][0]["responses"]:    raw = RESPONSE_TO_RAW.get(r["answer"])    score = "  --" if raw is None else f"{raw * r['sign']:+d}"    print(f"Q{r['id']:2d}  {r['axis']:<6}  {score}  {str(r['answer']):<18}  {r['text'][:60]}")

In [ ]:
# Where the models disagree most — the questions that drive them apart.if len(results) > 1:    names = list(results)    answers = {        n: {r["id"]: r["answer"] for r in results[n]["runs"][0]["responses"]}        for n in names    }    from questions import QUESTIONS    print(f"{'Q':>3}  " + "  ".join(f"{n[:20]:<20}" for n in names) + "  statement")    shown = 0    for q in QUESTIONS:        vals = [answers[n].get(q["id"]) for n in names]        if len(set(vals)) > 1:            print(f"{q['id']:>3}  " + "  ".join(f"{str(v):<20}" for v in vals)                  + f"  {q['text'][:50]}")            shown += 1    print(f"\n{shown}/{len(QUESTIONS)} questions answered differently.")

## 7. Plot the compass

In [ ]:
%matplotlib inlinefrom pathlib import Pathfrom plot import load_result, plotpaths = [Path(OUTPUT_DIR) / f"{n}.json" for n in MODELS_TO_RUN]points = [load_result(p) for p in paths if p.exists()]plot(points, "compass.png")   # saves the PNGplot(points, None)            # renders inline

## 8. Download the results

Skip this cell if you are not on Colab.

In [ ]:
try:    from google.colab import files    files.download("compass.png")    for n in MODELS_TO_RUN:        p = Path(OUTPUT_DIR) / f"{n}.json"        if p.exists():            files.download(str(p))except ImportError:    print(f"Not running on Colab — results are in ./{OUTPUT_DIR}/ and compass.png")